In [22]:
import importlib
import _4_ServingColumns
importlib.reload(_4_ServingColumns)

<module '_4_ServingColumns' from '/home/nakyung/projects/BDAIFin/5MODEL/STAGE1/_4_ServingColumns.py'>

In [ ]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:

from pathlib import Path
import json
import pandas as pd

TRAIN_IN = Path("../../5DATA/dataset/train_stage1")    # 24cols raw df parquet
TEST_IN  = Path("../../5DATA/dataset/test_stage1")     # 24cols raw df parquet

ART_OUT  = Path("../../5DATA/artifacts/stage1_artifacts_min.json")

TRAIN_OUT = Path("../../5DATA/dataset/TRAIN_stage1")
TEST_OUT  = Path("../../5DATA/dataset/TEST_stage1")

print("TRAIN_IN :", TRAIN_IN.resolve())
print("TEST_IN  :", TEST_IN.resolve())
print("ART_OUT  :", ART_OUT.resolve())
print("TRAIN_OUT:", TRAIN_OUT.resolve())
print("TEST_OUT :", TEST_OUT.resolve())

TRAIN_IN : /home/nakyung/projects/BDAIFin/5DATA/dataset/train_stage1
TEST_IN  : /home/nakyung/projects/BDAIFin/5DATA/dataset/test_stage1
ART_OUT  : /home/nakyung/projects/BDAIFin/5DATA/artifacts/stage1_artifacts_min.json
TRAIN_OUT: /home/nakyung/projects/BDAIFin/5DATA/dataset/TRAIN_stage1
TEST_OUT : /home/nakyung/projects/BDAIFin/5DATA/dataset/TEST_stage1


In [25]:
from _4_ServingColumns import (
    fit_stage1_artifacts_minimal,
    build_stage1_dataset_from_min_df,
)


In [ ]:

df_train_raw = pd.read_parquet(TRAIN_IN)
df_test_raw  = pd.read_parquet(TEST_IN)

print("train_raw:", df_train_raw.shape)
print("test_raw :", df_test_raw.shape)

display(df_train_raw.head(2))

train_raw: (609658, 24)
test_raw : (114209, 24)


,id,date,client_id,card_id,amount,merchant_id,mcc,is_online,fraud,has_error,...,err_bad_zipcode,err_insufficient_balance,err_technical_glitch,tx_year,tx_month,tx_day,tx_hour,weekday,is_refund,log_abs_amount
0,7475335,2010-01-01 00:14:00,1684,2140,26.459999,39021,4784,1,0,0,...,0,0,0,2010,1,1,0,4,0,3.312730
1,7475346,2010-01-01 00:34:00,394,4717,26.040001,39021,4784,1,0,0,...,0,0,0,2010,1,1,0,4,0,3.297317


In [27]:
# ===========================
# [Cell 4] Fit artifacts on TRAIN only + Save
# ===========================
artifacts = fit_stage1_artifacts_minimal(df_train_raw)

ART_OUT.parent.mkdir(parents=True, exist_ok=True)
with open(ART_OUT, "w", encoding="utf-8") as f:
    json.dump(artifacts, f, ensure_ascii=False, indent=2)

print("saved artifacts:", ART_OUT)
print(artifacts)

saved artifacts: ../../5DATA/artifacts/stage1_artifacts_min.json
{'base_rate': 0.01082246111754459, 'high_amount_q': 0.9, 'high_amount_thr': 4.9330339431762695, 'high_risk_days': [0, 4, 6]}


In [ ]:

train_feat = build_stage1_dataset_from_min_df(df_train_raw, artifacts)
test_feat  = build_stage1_dataset_from_min_df(df_test_raw, artifacts)

TRAIN_OUT.parent.mkdir(parents=True, exist_ok=True)
TEST_OUT.parent.mkdir(parents=True, exist_ok=True)

train_feat.to_parquet(TRAIN_OUT, index=False)
test_feat.to_parquet(TEST_OUT, index=False)

print("saved:", TRAIN_OUT, "shape:", train_feat.shape)
print("saved:", TEST_OUT,  "shape:", test_feat.shape)

display(train_feat.head(2))

/home/nakyung/projects/BDAIFin/5MODEL/STAGE1/_4_ServingColumns.py:143: RuntimeWarning: invalid value encountered in divide
  (amt_cumsum.to_numpy() / cnt_past.to_numpy()),
/home/nakyung/projects/BDAIFin/5MODEL/STAGE1/_4_ServingColumns.py:207: RuntimeWarning: invalid value encountered in divide
  (card_tx_1h_cumsum.to_numpy() / card_tx_cnt_past.to_numpy()),
/home/nakyung/projects/BDAIFin/5MODEL/STAGE1/_4_ServingColumns.py:143: RuntimeWarning: invalid value encountered in divide
  (amt_cumsum.to_numpy() / cnt_past.to_numpy()),
/home/nakyung/projects/BDAIFin/5MODEL/STAGE1/_4_ServingColumns.py:207: RuntimeWarning: invalid value encountered in divide
  (card_tx_1h_cumsum.to_numpy() / card_tx_cnt_past.to_numpy()),


saved: ../../5DATA/dataset/TRAIN_stage1 shape: (609658, 26)
saved: ../../5DATA/dataset/TEST_stage1 shape: (114209, 26)


,id,fraud,log_abs_amount,high_amount,amount_vs_client_avg_diff,amount_deviation,has_error,err_bad_cvv,err_bad_card_number,err_bad_expiration,...,tx_month,hour_cos,is_highrisk_weekday,seconds_since_prev_tx,card_velocity_spike_ratio,card_mcc_is_new,client_mcc_is_new,card_merchant_is_new,client_merchant_is_new,merchant_is_new_x_has_error
0,7497251,0,4.294970,0,0.00000,-0.151668,0,0,0,0,...,1,-0.965926,0,0.0,0.999999,1,1,1,1,0
1,7628167,0,2.833802,0,-0.90271,-1.294995,0,0,0,0,...,2,0.866025,1,2460780.0,0.999999,1,1,1,1,0


In [ ]:
LABEL = "fraud"

print("Train columns:", len(train_feat.columns))
print(train_feat.dtypes)

na_rate = (train_feat.isna().mean().sort_values(ascending=False).head(20))
print("\nTop NA rate:")
display(na_rate)

# feature list 확인
feat_cols = [c for c in train_feat.columns if c not in ["id", LABEL]]
print("\n#features:", len(feat_cols))
print(feat_cols)

Train columns: 26
id                               int64
fraud                             int8
log_abs_amount                 float32
high_amount                       int8
amount_vs_client_avg_diff      float32
amount_deviation               float32
has_error                         int8
err_bad_cvv                       int8
err_bad_card_number               int8
err_bad_expiration                int8
card_error_last1                  int8
client_error_last1                int8
card_fraud_last1                  int8
client_fraud_last1                int8
card_fraud_last3                  int8
tx_hour                           int8
tx_month                          int8
hour_cos                       float32
is_highrisk_weekday               int8
seconds_since_prev_tx          float32
card_velocity_spike_ratio      float32
card_mcc_is_new                   int8
client_mcc_is_new                 int8
card_merchant_is_new              int8
client_merchant_is_new            int8
merchan

id                           0.0
fraud                        0.0
log_abs_amount               0.0
high_amount                  0.0
amount_vs_client_avg_diff    0.0
amount_deviation             0.0
has_error                    0.0
err_bad_cvv                  0.0
err_bad_card_number          0.0
err_bad_expiration           0.0
card_error_last1             0.0
client_error_last1           0.0
card_fraud_last1             0.0
client_fraud_last1           0.0
card_fraud_last3             0.0
tx_hour                      0.0
tx_month                     0.0
hour_cos                     0.0
is_highrisk_weekday          0.0
seconds_since_prev_tx        0.0
dtype: float64


#features: 24
['log_abs_amount', 'high_amount', 'amount_vs_client_avg_diff', 'amount_deviation', 'has_error', 'err_bad_cvv', 'err_bad_card_number', 'err_bad_expiration', 'card_error_last1', 'client_error_last1', 'card_fraud_last1', 'client_fraud_last1', 'card_fraud_last3', 'tx_hour', 'tx_month', 'hour_cos', 'is_highrisk_weekday', 'seconds_since_prev_tx', 'card_velocity_spike_ratio', 'card_mcc_is_new', 'client_mcc_is_new', 'card_merchant_is_new', 'client_merchant_is_new', 'merchant_is_new_x_has_error']
